In [12]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"

for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")

if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [13]:
!pip install xgboost shap scikit-learn matplotlib pandas numpy scipy requests --quiet

In [14]:
# Configuration (mechanism analysis on UAVIDS-2025; single seed for reproducibility)
CONFIG = {
    "zenodo_record": "15336998",
    "data_dir": "data/uavids2025",
    "label_col": "label",
    "normal_value": "Normal Traffic",
    "drop_col_patterns": ["unnamed", "flowid", "srcaddr", "dstaddr",
                           "srcport", "dstport", "index", "timestamp"],
    "seed": 42,
    "n_shap": 2000,
    "normal_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.10, "test_shift": 0.10},
    "family_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.20},
    "xgb": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
            "subsample": 0.9, "colsample_bytree": 0.9, "tree_method": "hist"},
    "fig_dir": "figures",
    "report_dir": "reports",
}
for d in [CONFIG["data_dir"], CONFIG["fig_dir"], CONFIG["report_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG

{'zenodo_record': '15336998',
 'data_dir': 'data/uavids2025',
 'label_col': 'label',
 'normal_value': 'Normal Traffic',
 'drop_col_patterns': ['unnamed',
  'flowid',
  'srcaddr',
  'dstaddr',
  'srcport',
  'dstport',
  'index',
  'timestamp'],
 'seed': 42,
 'n_shap': 2000,
 'normal_fracs': {'train': 0.6,
  'cal': 0.2,
  'test_seen': 0.1,
  'test_shift': 0.1},
 'family_fracs': {'train': 0.6, 'cal': 0.2, 'test_seen': 0.2},
 'xgb': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.1,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
  'tree_method': 'hist'},
 'fig_dir': 'figures',
 'report_dir': 'reports'}

In [15]:
# Imports
import numpy as np, pandas as pd, requests, glob
import matplotlib.pyplot as plt
import xgboost as xgb, shap
from scipy.stats import spearmanr
from sklearn.metrics import balanced_accuracy_score
from src.data import load_csvs, detect_schema, prepare_splits
np.random.seed(CONFIG["seed"])
print("imports ready")

imports ready


In [16]:
# Ensure dataset present, then load
rec = CONFIG["zenodo_record"]
if not glob.glob(os.path.join(CONFIG["data_dir"], "**/*.csv"), recursive=True):
    meta = requests.get(f"https://zenodo.org/api/records/{rec}", timeout=60).json()
    for f in meta.get("files", []):
        name, url = f["key"], f["links"]["self"]
        if name.lower().endswith((".csv", ".zip", ".gz")):
            open(os.path.join(CONFIG["data_dir"], name), "wb").write(requests.get(url, timeout=600).content)
    for z in glob.glob(os.path.join(CONFIG["data_dir"], "*.zip")):
        import zipfile; zipfile.ZipFile(z).extractall(CONFIG["data_dir"])
df = load_csvs(CONFIG["data_dir"])
label_col, normal_value, families = detect_schema(df, CONFIG["label_col"], CONFIG["normal_value"])
print("shape:", df.shape, "| families:", families)

shape: (122171, 23) | families: ['Sybil Attack', 'Blackhole Attack', 'Wormhole Attack', 'Flooding Attack']


In [ ]:
# For each held-out family: train, measure the outcome (shift balanced accuracy), then
# explain it with SHAP. Transfer-fragility = attack-supporting attribution on seen attacks
# (positive SHAP) that reverses to Normal-supporting (negative SHAP) on the unseen family,
# weighted by how far it reverses:  sum_j max(0, seen_SHAP_j) * max(0, -heldout_SHAP_j).
def sample(A, n, seed):
    if len(A) > n:
        j = np.random.default_rng(seed).choice(len(A), n, replace=False); return A[j]
    return A

def mean_shap(expl, Xg):
    s = np.asarray(expl.shap_values(Xg))
    if s.ndim == 3:
        s = s[..., 1]
    return s.mean(0)

mech, store = [], {}
for F in families:
    S = prepare_splits(df, label_col, normal_value, F, CONFIG["drop_col_patterns"],
                       CONFIG["normal_fracs"], CONFIG["family_fracs"], CONFIG["seed"])
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=CONFIG["seed"], **CONFIG["xgb"]).fit(S["X_train"], S["y_train"])
    feats = S["feature_names"]

    pr_h = (clf.predict_proba(S["X_shift"])[:, 1] >= 0.5).astype(int)
    balacc = balanced_accuracy_score(S["y_shift"], pr_h)

    Xn = sample(S["X_seen"][S["fam_seen"] == normal_value], CONFIG["n_shap"], CONFIG["seed"])
    Xa = sample(S["X_seen"][np.isin(S["fam_seen"], S["seen_families"])], CONFIG["n_shap"], CONFIG["seed"])
    Xh = sample(S["X_shift"][S["fam_shift"] == F], CONFIG["n_shap"], CONFIG["seed"])

    expl = shap.TreeExplainer(clf)
    m_seen = mean_shap(expl, Xa)       # push toward "attack" for SEEN attacks
    m_held = mean_shap(expl, Xh)       # push for the UNSEEN family

    attack_support = np.maximum(m_seen, 0.0)      # pro-attack mass on seen attacks
    normal_pull    = np.maximum(-m_held, 0.0)     # pro-Normal mass on the unseen family
    flip = attack_support * normal_pull           # per-feature reversal contribution
    frag_raw = float(flip.sum())
    frag_norm = float(flip.sum() / (attack_support.sum() + 1e-12))

    mech.append({"held_out": F, "shift_balacc": round(balacc, 4),
                 "fragility": round(frag_raw, 4), "fragility_norm": round(frag_norm, 4)})
    store[F] = dict(feats=feats, m_seen=m_seen, m_held=m_held, flip=flip,
                    imp=np.abs(m_seen), Xn=Xn, Xa=Xa, Xh=Xh)
    print("done: %-18s balacc=%.3f  fragility=%.3f" % (F, balacc, frag_raw))

done: Sybil Attack       balacc=0.995  fragility=1.981


In [ ]:
# The mechanism: does transfer-fragility explain the accuracy ranking?
mt = pd.DataFrame(mech).sort_values("shift_balacc").reset_index(drop=True)
print(mt.to_string(index=False))
rho = spearmanr(mt["fragility"], mt["shift_balacc"]).correlation
print("\nSpearman(fragility, shift balanced accuracy) across families:", round(rho, 3),
      "  (strong negative => fragility predicts collapse)")
mt.to_csv(os.path.join(CONFIG["report_dir"], "05_mechanism_summary.csv"), index=False)

In [ ]:
# Per family: the features that reverse hardest (top reversal contribution)
frag_rows = []
for F in families:
    d = store[F]
    order = np.argsort(-d["flip"])[:8]
    for j in order:
        frag_rows.append({"held_out": F, "feature": d["feats"][j],
                          "seen_SHAP": round(float(d["m_seen"][j]), 4),
                          "heldout_SHAP": round(float(d["m_held"][j]), 4),
                          "reversal": round(float(d["flip"][j]), 4)})
    print("\n[%s]" % F)
    for j in order:
        print("   %-18s seen=%+.3f  held-out=%+.3f  reversal=%.3f"
              % (d["feats"][j], d["m_seen"][j], d["m_held"][j], d["flip"][j]))
pd.DataFrame(frag_rows).to_csv(os.path.join(CONFIG["report_dir"], "05_transfer_fragile_features.csv"), index=False)

In [ ]:
# Figure A: attribution transfer per family. On-diagonal = consistent reasoning;
# lower-right quadrant (pro-attack on seen, pro-Normal on unseen) = transfer-fragile.
n = len(families)
fig, ax = plt.subplots(1, n, figsize=(4.6 * n, 4.4))
if n == 1: ax = [ax]
mm = {r["held_out"]: r for r in mech}
for a, F in zip(ax, families):
    d = store[F]
    lim = max(np.abs(d["m_seen"]).max(), np.abs(d["m_held"]).max()) * 1.15 + 1e-6
    a.axhline(0, color="gray", lw=.5); a.axvline(0, color="gray", lw=.5)
    a.plot([-lim, lim], [-lim, lim], "--", color="gray", lw=.8)
    a.scatter(d["m_seen"], d["m_held"], s=18, alpha=.5, color="#c62828")
    for j in np.argsort(-d["imp"])[:5]:
        a.annotate(d["feats"][j], (d["m_seen"][j], d["m_held"][j]), fontsize=7)
    a.set_xlim(-lim, lim); a.set_ylim(-lim, lim)
    a.set_xlabel("mean SHAP, seen attacks"); a.set_ylabel("mean SHAP, held-out family")
    a.set_title("%s\nfragility=%.2f  bal-acc=%.2f"
                % (str(F).replace(" Attack", ""), mm[F]["fragility"], mm[F]["shift_balacc"]))
fig.suptitle("Attribution transfer: lower-right = transfer-fragile features drive the collapse")
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["fig_dir"], "05_attribution_transfer.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure B: for the family that collapses most, the single hardest-reversing feature.
# Shows the unseen family sitting with Normal rather than with known attacks.
Fc = mt.iloc[0]["held_out"]
d = store[Fc]
jf = int(np.argsort(-d["flip"])[0])
feat = d["feats"][jf]
plt.figure(figsize=(7, 4))
plt.hist(d["Xn"][:, jf], bins=40, density=True, alpha=.5, label="Normal", color="#1565c0")
plt.hist(d["Xa"][:, jf], bins=40, density=True, alpha=.5, label="seen attacks", color="#2e7d32")
plt.hist(d["Xh"][:, jf], bins=40, density=True, alpha=.5,
         label="held-out %s" % str(Fc).replace(" Attack", ""), color="#c62828")
plt.xlabel("%s (standardized)" % feat); plt.ylabel("density")
plt.title("Hardest-reversing feature for %s" % str(Fc).replace(" Attack", ""))
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(CONFIG["fig_dir"], "05_fragile_feature_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit results (end-of-unit discipline)
!git add reports/ figures/ notebooks/
!git commit -m "05 UAVIDS-2025: transfer-fragile-feature mechanism via sign-flip fragility score (replaces cosine)"
!git push origin main